# Assignment 2 — Parts B & C: Learned Embeddings & Probing
**Course**: Introduction to Natural Language Processing  
**Topic**: Social Media Sentiment Analysis — Word Representations (Phase 2)  
**Team**: Team Alpha (`nlp-project-teamalpha`)  

This notebook covers:
- **Task B1**: Word2Vec Continuous Bag-of-Words (CBOW, sg=0) Training.
- **Task B2**: Word2Vec Skip-gram (sg=1) Training with Identical Hyperparameters.
- **Task B3**: Theoretical Comparison of CBOW vs. Skip-gram Objectives.
- **Task B4**: fastText Training & Demonstration of Sub-word Out-of-Vocabulary (OOV) Generalization.
- **Task C1**: Top-N Nearest Semantic Neighbors across Models.
- **Task C2**: Pairwise Cosine Similarity (Expected Similar vs. Unrelated).
- **Task C3**: Vector Analogy Probing ($a - b + c \approx d$) & Corpus Size Discussion.
- **Task C4**: Comprehensive Comparison: Count-based vs. Learned Representations.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec, FastText

# Load tokenized sentences
data_path = "../data/cleaned_corpus.csv" if os.path.exists("../data/cleaned_corpus.csv") else "data/cleaned_corpus.csv"
df = pd.read_csv(data_path)
sentences = [str(text).split() for text in df['cleaned_text'].fillna('') if len(str(text).strip()) > 0]
print(f"Prepared {len(sentences):,} tokenized sentences.")


## Part B: Training Learned Embeddings
All three models are trained using **identical hyperparameters**: `vector_size=100`, `window=5`, `min_count=2`, `epochs=30`, `seed=42`.


In [ ]:
# B1. Train Word2Vec CBOW
w2v_cbow = Word2Vec(sentences, vector_size=100, window=5, min_count=2, epochs=30, sg=0, seed=42, workers=4)

# B2. Train Word2Vec Skip-gram
w2v_sg = Word2Vec(sentences, vector_size=100, window=5, min_count=2, epochs=30, sg=1, seed=42, workers=4)

# B4. Train fastText
ft_model = FastText(sentences, vector_size=100, window=5, min_count=2, epochs=30, min_n=3, max_n=6, seed=42, workers=4)

print(f"CBOW Vocab:      {len(w2v_cbow.wv.key_to_index):,} tokens")
print(f"Skip-gram Vocab: {len(w2v_sg.wv.key_to_index):,} tokens")
print(f"fastText Vocab:  {len(ft_model.wv.key_to_index):,} tokens")


## Task B4 Demonstration: Out-of-Vocabulary / Misspelled Word Resilience
Word2Vec fails with `KeyError` on unseen words, whereas fastText uses character $n$-grams to construct meaningful representations.


In [ ]:
test_oov_words = ["googl", "microsofft", "iphne", "batry"]

for word in test_oov_words:
    print(f"\nQuery Misspelled Word: '{word}'")
    # Word2Vec lookup:
    if word in w2v_sg.wv:
        print(f"  Word2Vec: In vocab")
    else:
        print(f"  Word2Vec: KeyError (Out-of-Vocabulary -> Crashes)")
    
    # fastText lookup:
    sims = ft_model.wv.most_similar(word, topn=3)
    sim_str = ", ".join([f"{w} ({s:.3f})" for w, s in sims])
    print(f"  fastText: Success! Inferred from sub-words -> Top Neighbors: {sim_str}")


## Task C1: Top-N Nearest Semantic Neighbors
Inspecting domain words across the 3 embedding spaces.


In [ ]:
query_words = ["apple", "google", "phone", "battery"]
models = {"CBOW": w2v_cbow.wv, "Skip-gram": w2v_sg.wv, "fastText": ft_model.wv}

for q in query_words:
    print(f"\nQuery: '{q}'")
    for name, m_wv in models.items():
        if q in m_wv:
            top_words = m_wv.most_similar(q, topn=4)
            print(f"  {name:<10s}: {', '.join([f'{w} ({s:.3f})' for w, s in top_words])}")


## Task C2: Pairwise Cosine Similarity
Comparing expected similar pairs against expected unrelated pairs.


In [ ]:
pairs = [
    ("phone", "iphone", "Expected Similar"),
    ("google", "android", "Expected Similar"),
    ("apple", "battery", "Expected Unrelated"),
    ("phone", "traffic", "Expected Unrelated")
]

table = []
for w1, w2, label in pairs:
    table.append({
        "Pair": f"{w1} <-> {w2}",
        "CBOW": w2v_cbow.wv.similarity(w1, w2),
        "Skip-gram": w2v_sg.wv.similarity(w1, w2),
        "fastText": ft_model.wv.similarity(w1, w2),
        "Label": label
    })

pd.DataFrame(table)


## Task C3: Vector Analogy Probing ($a - b + c \approx d$)
Testing `'iphone' - 'apple' + 'google' \approx 'android'` and discussing corpus size requirements.


In [ ]:
for name, m_wv in models.items():
    try:
        res = m_wv.most_similar(positive=['iphone', 'google'], negative=['apple'], topn=3)
        print(f"{name:<10s}: {', '.join([f'{w} ({s:.3f})' for w, s in res])}")
    except Exception as e:
        print(f"{name:<10s}: Error: {e}")
